# Cuaderno Anexo — Clasificación de Texto con BoW y TF-IDF

> **Nota:** Este cuaderno es un material complementario. Retoma los conceptos de BoW y TF-IDF del cuaderno de síntesis y los aplica a una tarea de clasificación supervisada. No forma parte de la secuencia obligatoria del módulo, pero es un buen punto de partida para quien quiera profundizar en NLP aplicado.

## Objetivo

Entender cómo las representaciones de texto (BoW, TF-IDF) se conectan con modelos de aprendizaje automático para clasificar textos automáticamente. Se introduce el concepto de Pipeline de scikit-learn como herramienta de integración.

## Resultados de aprendizaje

Al terminar este cuaderno vas a poder:

- Construir un Pipeline que encadene vectorización y clasificación.
- Entrenar y evaluar un clasificador de texto con Naive Bayes y Regresión Logística.
- Interpretar las métricas de evaluación (precision, recall, F1).
- Identificar las limitaciones de este enfoque para textos con poca semántica compartida.

## Relación con cuadernos anteriores

Este cuaderno usa BoW y TF-IDF (introducidos en el cuaderno de síntesis) como entrada para modelos de clasificación. Es un paso natural hacia el uso de embeddings en tareas supervisadas.


## Microglosario

| Término | Definición |
|---|---|
| **Clasificación supervisada** | Tarea de aprendizaje automático donde el modelo aprende de ejemplos etiquetados para predecir la clase de nuevos textos. |
| **Pipeline** | Cadena de pasos (vectorización + modelo) que se ejecutan en orden y se tratan como una sola unidad. |
| **Naive Bayes** | Modelo probabilístico simple y rápido, muy efectivo para clasificación de texto. |
| **Regresión Logística** | Modelo lineal que aprende pesos para cada palabra del vocabulario y predice probabilidades de clase. |
| **Precision** | De todos los que predije como clase X, ¿cuántos realmente eran clase X? |
| **Recall** | De todos los que realmente son clase X, ¿cuántos encontré? |
| **F1-score** | Media armónica entre precision y recall. |


## 1. Importaciones


In [ ]:
# Operaciones numéricas
import numpy as np

# Vectorización
from sklearn.feature_extraction.text import TfidfVectorizer

# División del dataset en entrenamiento y prueba
from sklearn.model_selection import train_test_split

# Pipeline para encadenar pasos
from sklearn.pipeline import Pipeline

# Modelos de clasificación
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Métricas de evaluación
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Visualización
import matplotlib.pyplot as plt

print("Librerías importadas correctamente.")


## 2. Datos de ejemplo

Vamos a usar un corpus pequeño y controlado de reseñas positivas y negativas. Los textos están en español rioplatense para que el modelo capture vocabulario local.

> **Para mirar:** ¿cuántos textos de entrenamiento y cuántos de prueba tenemos? ¿Es un número razonable para entrenar un modelo? ¿Por qué sí o por qué no?


In [ ]:
# Corpus de ejemplo: reseñas en español rioplatense
# Etiqueta 1 = Positivo, Etiqueta 0 = Negativo
textos = [
    "La milanesa a caballo estaba espectacular",
    "Qué buena onda la atención en el bar",
    "El flan con dulce de leche es lo más",
    "El bife de chorizo llegó frío y duro",
    "Mucho quilombo, tardaron una banda en traer la cuenta",
    "La verdad, la pizza dejaba bastante que desear",
]

# Etiquetas: 1 para Positivo, 0 para Negativo
etiquetas = np.array([1, 1, 1, 0, 0, 0])

# Mostramos los textos con sus etiquetas
print("Corpus de ejemplo:")
for i in range(len(textos)):
    texto = textos[i]
    etiqueta = "Positivo" if etiquetas[i] == 1 else "Negativo"
    print(f"  [{etiqueta}] {texto}")


## 3. División en entrenamiento y prueba

Dividimos el corpus en dos partes:
- **Entrenamiento**: el modelo aprende de estos textos.
- **Prueba**: evaluamos el modelo en textos que nunca vio.

> **Nota:** con 6 textos el conjunto es demasiado pequeño para ser estadísticamente significativo. En la práctica, un clasificador necesita cientos o miles de ejemplos. Este dataset es solo para ilustrar el flujo de trabajo.


In [ ]:
# Dividimos los datos en entrenamiento (67%) y prueba (33%)
# 'test_size=0.33' indica que el 33% va a prueba
# 'random_state=42' fija la semilla para reproducibilidad
textos_entrenamiento, textos_prueba, etiquetas_entrenamiento, etiquetas_prueba = train_test_split(
    textos, etiquetas, test_size=0.33, random_state=42
)

cantidad_entrenamiento = len(textos_entrenamiento)
cantidad_prueba = len(textos_prueba)

print(f"Textos de entrenamiento: {cantidad_entrenamiento}")
print(f"Textos de prueba: {cantidad_prueba}")


## 4. Construcción del Pipeline

Un Pipeline encadena dos pasos que siempre van juntos:
1. **Vectorización** (TF-IDF): convierte los textos en números.
2. **Clasificación** (Naive Bayes o Regresión Logística): aprende a predecir la clase.

La ventaja es que el Pipeline garantiza que la vectorización se aprende **solo** sobre los datos de entrenamiento y se aplica de la misma manera a los de prueba, evitando filtración de información entre conjuntos.

> **Para mirar:** ¿qué diferencia ves en la salida entre Naive Bayes y Regresión Logística? ¿Uno funciona mejor que el otro en este caso?


In [ ]:
# Pipeline 1: TF-IDF + Naive Bayes
# 'Pipeline' encadena el vectorizador y el clasificador
pipeline_naive_bayes = Pipeline([
    ('vectorizador', TfidfVectorizer()),
    ('clasificador', MultinomialNB()),
])

# Entrenamos el pipeline con los datos de entrenamiento
# 'fit' aprende el vocabulario TF-IDF Y los pesos del clasificador
pipeline_naive_bayes.fit(textos_entrenamiento, etiquetas_entrenamiento)

# Predecimos las clases de los textos de prueba
predicciones_nb = pipeline_naive_bayes.predict(textos_prueba)

print("Pipeline Naive Bayes — Resultados en datos de prueba:")
print()
print(classification_report(etiquetas_prueba, predicciones_nb,
                             target_names=["Negativo", "Positivo"]))


In [ ]:
# Pipeline 2: TF-IDF + Regresión Logística
pipeline_logistica = Pipeline([
    ('vectorizador', TfidfVectorizer()),
    ('clasificador', LogisticRegression(max_iter=1000)),
])

# Entrenamos y predecimos
pipeline_logistica.fit(textos_entrenamiento, etiquetas_entrenamiento)
predicciones_lr = pipeline_logistica.predict(textos_prueba)

print("Pipeline Regresión Logística — Resultados en datos de prueba:")
print()
print(classification_report(etiquetas_prueba, predicciones_lr,
                             target_names=["Negativo", "Positivo"]))


## 5. Matriz de confusión

La matriz de confusión muestra visualmente qué acertó y qué erró el modelo:
- **Diagonal principal**: predicciones correctas.
- **Fuera de la diagonal**: errores (lo que el modelo confundió).

> **Para mirar:** ¿el modelo confunde más los positivos con negativos o los negativos con positivos? ¿Por qué puede pasar eso?


In [ ]:
# Calculamos la matriz de confusión para Naive Bayes
matriz_confusion = confusion_matrix(etiquetas_prueba, predicciones_nb)

# Visualizamos con ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(6, 5))
display_confusion = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusion,
    display_labels=["Negativo", "Positivo"]
)
display_confusion.plot(ax=ax, colorbar=False)

ax.set_title("Matriz de Confusión — Naive Bayes", fontweight="bold", loc="left")
plt.tight_layout()
plt.show()


## Reflexión y limitaciones

### ¿Qué captura este modelo?

El Pipeline con TF-IDF aprende qué palabras están asociadas a cada clase. Por ejemplo, aprende que "espectacular", "buena" y "lo más" tienden a aparecer en reseñas positivas, y "frío", "quilombo" y "desear" en negativas.

### ¿Qué no captura?

- **Negación**: "no me gustó" y "me gustó" tienen palabras casi iguales pero significados opuestos.
- **Ironía y sarcasmo**: "qué maravilla, llegó fría la comida" sería clasificado como positivo.
- **Vocabulario nuevo**: si aparece una palabra que no estaba en el entrenamiento, el modelo no sabe qué hacer con ella.

### ¿Qué alternativa existe?

Usar embeddings en lugar de TF-IDF como representación mejora muchos de estos problemas, especialmente el de vocabulario nuevo y el de significado semántico.


## Actividad de cierre

1. Agregá al corpus 4 textos nuevos (2 positivos y 2 negativos) con vocabulario diferente.
2. Volvé a entrenar el Pipeline y observá si el rendimiento cambia.
3. Intentá escribir un texto ambiguo (que podría ser positivo o negativo) y fijate cómo lo clasifica el modelo. ¿Qué dice la probabilidad predicha?

> **Pista:** podés obtener probabilidades (en lugar de solo la clase predicha) con `pipeline.predict_proba(textos_nuevos)`.


In [ ]:
# Usá esta celda para la actividad de cierre
